In [17]:
# Necessary libraries
import pandas as pd
import numpy as np
from datetime import datetime
from pandas.errors import OutOfBoundsDatetime
import re


In [ ]:
# Create parser of data

def parse_nominal_predictions(file_path):
    """
    Parse the Nominal prediction Excel sheet and restructure into clean format.
    
    Parameters:
    -----------
    file_path : str
        Path to the Excel file or CSV
    
    Returns:
    --------
    pd.DataFrame
        Restructured data with columns: Date, Actual_Spot, Lag_Quarters, Predicted_Spot
    """
    
    # Read the Excel file
    # Use header=None to get all rows as data
    df = pd.read_excel("Project Japan 202 backtesting.xls", sheet_name='Nom $ apprec method a)12 months', header=None)
    
    # Initialize lists to store parsed data
    dates = []
    actual_spots = []
    lag_quarters = []
    predicted_spots = []
    
    # Find the date and spot rate columns (should be columns 0 and 1)
    date_col = 0
    spot_col = 1
    
    # Iterate through rows to find data
    for idx, row in df.iterrows():
        # Get date and actual spot rate
        date_val = row[date_col]
        spot_val = row[spot_col]
        
        # Skip if date is not valid
        if pd.isna(date_val) or not isinstance(date_val, (datetime, pd.Timestamp, str)):
            continue
            
        # Skip if spot rate is not numeric
        if pd.isna(spot_val):
            continue
            
        try:
            spot_val = float(spot_val)
        except (ValueError, TypeError):
            continue
        
        # Now look for N/360 and E(S¥/$N) pairs in this row
        # They appear in clusters across the columns
        for col_idx in range(2, len(row)):
            cell_val = row[col_idx]
            
            # Check if this cell contains N/360 value (lag in quarters)
            if pd.notna(cell_val):
                try:
                    # If it's a number, check if next few columns have E(S¥/$N)
                    n_val = float(cell_val)
                    
                    # N/360 represents quarters, e.g., 3 = 3 quarters
                    # Look ahead for the predicted spot rate
                    # Typically E(S¥/$N) appears 1-2 columns after N/360
                    for offset in range(1, 4):
                        if col_idx + offset < len(row):
                            pred_val = row[col_idx + offset]
                            if pd.notna(pred_val):
                                try:
                                    pred_spot = float(pred_val)
                                    # Only accept reasonable spot rate values (50-200 range for ¥/$)
                                    if 50 <= pred_spot <= 200:
                                        dates.append(date_val)
                                        actual_spots.append(spot_val)
                                        lag_quarters.append(int(n_val))
                                        predicted_spots.append(pred_spot)
                                        break
                                except (ValueError, TypeError):
                                    continue
                except (ValueError, TypeError):
                    continue
    
    # Create DataFrame
    result_df = pd.DataFrame({
        'Date': dates,
        'Actual_Spot': actual_spots,
        'Lag_Quarters': lag_quarters,
        'Predicted_Spot': predicted_spots
    })
    
    # Remove duplicates
    result_df = result_df.drop_duplicates()
    
    # Sort by date and lag
    result_df = result_df.sort_values(['Date', 'Lag_Quarters'])
    
    return result_df


def parse_nominal_predictions_manual(file_path):
    """
    Alternative parser with manual column specification.
    Use this if the automatic parser doesn't work correctly.
    """
    
    # Read the file
    df = pd.read_excel("Project Japan 202 backtesting.xls", sheet_name='Nom $ apprec method a)12 months', header=None)
    
    # Manually specify where predictions are
    # Based on your data, we need to identify the column groups
    
    # This will need adjustment based on your actual file structure
    # Example structure:
    prediction_groups = [
        # (n_quarters_col, predicted_spot_col)
        (2, 3),   # First prediction group
        (5, 6),   # Second prediction group  
        # Add more as needed
    ]
    
    dates = []
    actual_spots = []
    predictions = []
    
    for idx, row in df.iterrows():
        date_val = row[0]
        spot_val = row[1]
        
        if pd.isna(date_val) or pd.isna(spot_val):
            continue
            
        try:
            spot_val = float(spot_val)
        except:
            continue
        
        for n_col, pred_col in prediction_groups:
            if pd.notna(row[n_col]) and pd.notna(row[pred_col]):
                try:
                    n_quarters = int(float(row[n_col]))
                    pred_spot = float(row[pred_col])
                    
                    dates.append(date_val)
                    actual_spots.append(spot_val)
                    predictions.append({
                        'Lag_Quarters': n_quarters,
                        'Predicted_Spot': pred_spot
                    })
                except:
                    continue
    
    # Flatten into DataFrame
    rows = []
    for date, spot, pred in zip(dates, actual_spots, predictions):
        rows.append({
            'Date': date,
            'Actual_Spot': spot,
            'Lag_Quarters': pred['Lag_Quarters'],
            'Predicted_Spot': pred['Predicted_Spot']
        })
    
    return pd.DataFrame(rows)


def calculate_forecast_errors(df):
    """
    Calculate forecast errors for each prediction.
    Error = Actual Future Spot - Predicted Spot
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with columns: Date, Actual_Spot, Lag_Quarters, Predicted_Spot
    
    Returns:
    --------
    pd.DataFrame
        Original data with added columns: Future_Date, Future_Spot, Forecast_Error
    """
    
    # Ensure Date column is datetime
    df['Date'] = pd.to_datetime(df['Date'])
    
    # Sort by date
    df = df.sort_values('Date').reset_index(drop=True)
    
    # Create a mapping of dates to actual spot rates
    spot_rate_map = dict(zip(df['Date'], df['Actual_Spot']))
    
    # Calculate future dates and get actual future spot rates
    future_dates = []
    future_spots = []
    forecast_errors = []
    
    for _, row in df.iterrows():
        # Calculate future date (add lag_quarters * 3 months)
        future_date = row['Date'] + pd.DateOffset(months=row['Lag_Quarters'] * 3)
        
        # Get actual spot rate at future date
        if future_date in spot_rate_map:
            future_spot = spot_rate_map[future_date]
            # Error = Actual - Predicted
            error = future_spot - row['Predicted_Spot']
        else:
            future_spot = np.nan
            error = np.nan
        
        future_dates.append(future_date)
        future_spots.append(future_spot)
        forecast_errors.append(error)
    
    # Add to DataFrame
    df['Future_Date'] = future_dates
    df['Future_Spot'] = future_spots
    df['Forecast_Error'] = forecast_errors
    
    # Remove rows where we don't have future data
    df = df.dropna(subset=['Forecast_Error'])
    
    return df


def analyze_by_lag(df):
    """
    Calculate standard deviation of forecast errors for each lag period.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with Forecast_Error column
    
    Returns:
    --------
    pd.DataFrame
        Summary statistics by lag period
    """
    
    summary = df.groupby('Lag_Quarters').agg({
        'Forecast_Error': ['count', 'mean', 'std', lambda x: np.sqrt(np.mean(x**2))]
    }).round(4)
    
    summary.columns = ['Count', 'Mean_Error', 'Std_Dev', 'RMSE']
    summary = summary.reset_index()
    
    return summary


# Example usage:
if __name__ == "__main__":
    # Parse the data
    file_path = "Project Japan 202 backtesting.xls"  
    
    print("Parsing nominal predictions...")
    df = parse_nominal_predictions("Project Japan 202 backtesting.xls")
    
    print(f"\nParsed {len(df)} prediction records")
    print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
    print(f"Lag quarters found: {sorted(df['Lag_Quarters'].unique())}")
    
    # Calculate forecast errors
    print("\nCalculating forecast errors...")
    df_errors = calculate_forecast_errors(df)
    
    print(f"\n{len(df_errors)} predictions with calculable errors")
    
    # Analyze by lag
    print("\nStandard deviation by lag period:")
    summary = analyze_by_lag(df_errors)
    print(summary)
    
    # Find optimal lag
    optimal_lag = summary.loc[summary['Std_Dev'].idxmin()]
    print(f"\nOptimal lag period: {optimal_lag['Lag_Quarters']} quarters")
    print(f"Standard deviation: {optimal_lag['Std_Dev']:.4f}")
    
    # Save cleaned data
    df_errors.to_csv("nominal_predictions_cleaned.csv", index=False)
    summary.to_csv("nominal_summary_by_lag.csv", index=False)
    print("\nData saved to CSV files")

In [23]:
# Create Parser

def parse_nominal_predictions(file_path):
    """
    Parse the Nominal prediction Excel sheet and restructure into clean format.
    
    Parameters:
    -----------
    file_path : str
        Path to the Excel file or CSV
    
    Returns:
    --------
    pd.DataFrame
        Restructured data with columns: Date, Actual_Spot, Lag_Quarters, Predicted_Spot
    """
    
    # Read the Excel file
    # Use header=None to get all rows as data
    df = pd.read_excel("Project Japan 202 backtesting.xls", sheet_name='Nom $ apprec method a)12 months', header=None)
    
    # Initialize lists to store parsed data
    dates = []
    actual_spots = []
    lag_quarters = []
    predicted_spots = []
    
    # Find the date and spot rate columns (should be columns 0 and 1)
    date_col = 0
    spot_col = 1
    
    # Iterate through rows to find data
    for idx, row in df.iterrows():
        # Get date and actual spot rate
        date_val = row[date_col]
        spot_val = row[spot_col]
        
        # Skip if date is not valid
        if pd.isna(date_val) or not isinstance(date_val, (datetime, pd.Timestamp, str)):
            continue
            
        # Skip if spot rate is not numeric
        if pd.isna(spot_val):
            continue
            
        try:
            spot_val = float(spot_val)
        except (ValueError, TypeError):
            continue
        
        # Now look for N/360 and E(S¥/$N) pairs in this row
        # They appear in clusters across the columns
        for col_idx in range(2, len(row)):
            cell_val = row[col_idx]
            
            # Check if this cell contains N/360 value (lag in quarters)
            if pd.notna(cell_val):
                try:
                    # If it's a number, check if next few columns have E(S¥/$N)
                    n_val = float(cell_val)
                    
                    # N/360 represents quarters, e.g., 3 = 3 quarters
                    # Only accept reasonable lag values (1-12 quarters as specified)
                    if 1 <= n_val <= 12:
                        # Look ahead for the predicted spot rate
                        # Typically E(S¥/$N) appears 1-2 columns after N/360
                        for offset in range(1, 4):
                            if col_idx + offset < len(row):
                                pred_val = row[col_idx + offset]
                                if pd.notna(pred_val):
                                    try:
                                        pred_spot = float(pred_val)
                                        # Only accept reasonable spot rate values (50-200 range for ¥/$)
                                        if 50 <= pred_spot <= 200:
                                            dates.append(date_val)
                                            actual_spots.append(spot_val)
                                            lag_quarters.append(int(n_val))
                                            predicted_spots.append(pred_spot)
                                            break
                                    except (ValueError, TypeError):
                                        continue
                except (ValueError, TypeError):
                    continue
    
    # Create DataFrame
    result_df = pd.DataFrame({
        'Date': dates,
        'Actual_Spot': actual_spots,
        'Lag_Quarters': lag_quarters,
        'Predicted_Spot': predicted_spots
    })
    
    # Remove duplicates
    result_df = result_df.drop_duplicates()
    
    # Sort by date and lag
    result_df = result_df.sort_values(['Date', 'Lag_Quarters'])
    
    return result_df


def parse_nominal_predictions_manual(file_path):
    """
    Alternative parser with manual column specification.
    Use this if the automatic parser doesn't work correctly.
    """
    
    # Read the file
    df = pd.read_excel("Project Japan 202 backtesting.xls", sheet_name='Nom $ apprec method a)12 months', header=None)
    
    # Manually specify where predictions are
    # Based on your data, we need to identify the column groups
    
    # This will need adjustment based on your actual file structure
    # Example structure:
    prediction_groups = [
        # (n_quarters_col, predicted_spot_col)
        (2, 3),   # First prediction group
        (5, 6),   # Second prediction group  
        # Add more as needed
    ]
    
    dates = []
    actual_spots = []
    predictions = []
    
    for idx, row in df.iterrows():
        date_val = row[0]
        spot_val = row[1]
        
        if pd.isna(date_val) or pd.isna(spot_val):
            continue
            
        try:
            spot_val = float(spot_val)
        except:
            continue
        
        for n_col, pred_col in prediction_groups:
            if pd.notna(row[n_col]) and pd.notna(row[pred_col]):
                try:
                    n_quarters = int(float(row[n_col]))
                    pred_spot = float(row[pred_col])
                    
                    dates.append(date_val)
                    actual_spots.append(spot_val)
                    predictions.append({
                        'Lag_Quarters': n_quarters,
                        'Predicted_Spot': pred_spot
                    })
                except:
                    continue
    
    # Flatten into DataFrame
    rows = []
    for date, spot, pred in zip(dates, actual_spots, predictions):
        rows.append({
            'Date': date,
            'Actual_Spot': spot,
            'Lag_Quarters': pred['Lag_Quarters'],
            'Predicted_Spot': pred['Predicted_Spot']
        })
    
    return pd.DataFrame(rows)


def calculate_forecast_errors(df):
    """
    Calculate forecast errors for each prediction.
    Error = Actual Future Spot - Predicted Spot
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with columns: Date, Actual_Spot, Lag_Quarters, Predicted_Spot
    
    Returns:
    --------
    pd.DataFrame
        Original data with added columns: Future_Date, Future_Spot, Forecast_Error
    """
    
    # Ensure Date column is datetime
    df['Date'] = pd.to_datetime(df['Date'])
    
    # Sort by date
    df = df.sort_values('Date').reset_index(drop=True)
    
    # Create a mapping of dates to actual spot rates
    spot_rate_map = dict(zip(df['Date'], df['Actual_Spot']))
    
    # Calculate future dates and get actual future spot rates
    future_dates = []
    future_spots = []
    forecast_errors = []
    
    for _, row in df.iterrows():
        # Validate lag_quarters is reasonable (1-12)
        if not (1 <= row['Lag_Quarters'] <= 12):
            print(f"Warning: Skipping invalid lag value: {row['Lag_Quarters']}")
            continue
            
        try:
            # Calculate future date (add lag_quarters * 3 months)
            future_date = row['Date'] + pd.DateOffset(months=int(row['Lag_Quarters']) * 3)
            
            # Get actual spot rate at future date
            if future_date in spot_rate_map:
                future_spot = spot_rate_map[future_date]
                # Error = Actual - Predicted
                error = future_spot - row['Predicted_Spot']
            else:
                future_spot = np.nan
                error = np.nan
            
            future_dates.append(future_date)
            future_spots.append(future_spot)
            forecast_errors.append(error)
        except (OutOfBoundsDatetime, OverflowError) as e:
            print(f"Warning: Date overflow for lag {row['Lag_Quarters']} quarters from {row['Date']}")
            continue
    
    # Add to DataFrame
    df['Future_Date'] = future_dates
    df['Future_Spot'] = future_spots
    df['Forecast_Error'] = forecast_errors
    
    # Remove rows where we don't have future data
    df = df.dropna(subset=['Forecast_Error'])
    
    return df


def analyze_by_lag(df):
    """
    Calculate standard deviation of forecast errors for each lag period.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with Forecast_Error column
    
    Returns:
    --------
    pd.DataFrame
        Summary statistics by lag period
    """
    
    summary = df.groupby('Lag_Quarters').agg({
        'Forecast_Error': ['count', 'mean', 'std', lambda x: np.sqrt(np.mean(x**2))]
    }).round(4)
    
    summary.columns = ['Count', 'Mean_Error', 'Std_Dev', 'RMSE']
    summary = summary.reset_index()
    
    return summary


# Example usage:
if __name__ == "__main__":
    # Parse the data
    file_path = "Project Japan 202 backtesting.xls"  # Update with your file path
    
    print("Parsing nominal predictions...")
    df = parse_nominal_predictions("Project Japan 202 backtesting.xls")
    
    print(f"\nParsed {len(df)} prediction records")
    print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
    print(f"Lag quarters found: {sorted(df['Lag_Quarters'].unique())}")
    
    # Calculate forecast errors
    print("\nCalculating forecast errors...")
    df_errors = calculate_forecast_errors(df)
    
    print(f"\n{len(df_errors)} predictions with calculable errors")
    
    # Analyze by lag
    print("\nStandard deviation by lag period:")
    summary = analyze_by_lag(df_errors)
    print(summary)
    
    # Find optimal lag
    optimal_lag = summary.loc[summary['Std_Dev'].idxmin()]
    print(f"\nOptimal lag period: {optimal_lag['Lag_Quarters']} quarters")
    print(f"Standard deviation: {optimal_lag['Std_Dev']:.4f}")
    
    # Save cleaned data
    df_errors.to_csv("nominal_predictions_cleaned.csv", index=False)
    summary.to_csv("nominal_summary_by_lag.csv", index=False)
    print("\nData saved to CSV files")

Parsing nominal predictions...

Parsed 213 prediction records
Date range: 1995-12-31 00:00:00 to 2019-12-31 00:00:00
Lag quarters found: [1, 2, 3, 4, 5, 6, 7, 11]

Calculating forecast errors...

141 predictions with calculable errors

Standard deviation by lag period:
   Lag_Quarters  Count  Mean_Error  Std_Dev     RMSE
0             1     70      0.4229  22.8728  22.7128
1             2     47     -4.1310  36.2825  36.1313
2             3     20     -7.5207  49.1294  48.4724
3             4      1    -17.6389      NaN  17.6389
4             5      1     11.9600      NaN  11.9600
5             7      1    -36.3879      NaN  36.3879
6            11      1     16.9813      NaN  16.9813

Optimal lag period: 1.0 quarters
Standard deviation: 22.8728

Data saved to CSV files
